# 🏔️ Collection Monitor (Current Job)
Real-time monitoring of the active collection plan, logs, and captured images.

### 📊 Collection Log (Last 10 Events)

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Image, HTML
import ipywidgets as widgets

log_path = Path('data/collection.log')

def get_recent_logs(n=10):
    if not log_path.exists():
        return pd.DataFrame()
    
    entries = []
    with open(log_path, 'r') as f:
        lines = f.readlines()
        for line in lines[-n:]:
            try:
                entry = json.loads(line.strip())
                # Flatten metadata for display
                meta = entry.pop('metadata', {})
                entry.update(meta)
                entries.append(entry)
            except:
                pass
    
    df = pd.DataFrame(entries)
    if not df.empty:
        # Cleanup timestamp for readability
        df['timestamp'] = pd.to_datetime(df['timestamp']).dt.strftime('%H:%M:%S')
    return df

display(get_recent_logs())

### 🖼️ Job Capture Browser

In [ ]:
def get_job_captures():
    """Gets captures associated with the current PLAN in the logs."""
    if not log_path.exists():
        return []
    
    captures = []
    with open(log_path, 'r') as f:
        for line in f:
            try:
                entry = json.loads(line.strip())
                if entry.get('event') == 'CAPTURE' and entry.get('status') == 'SUCCESS':
                    meta = entry.get('metadata', {})
                    if 'image_path' in meta:
                        captures.append({
                            'timestamp': entry['timestamp'],
                            'path': meta['image_path'],
                            'step': meta.get('step_index', 'N/A'),
                            'metar': meta.get('metar_path')
                        })
            except:
                pass
    return sorted(captures, key=lambda x: x['timestamp'], reverse=True)

job_caps = get_job_captures()

dropdown = widgets.Dropdown(
    options=[(f"Step {c['step']} - {c['timestamp']}", c) for c in job_caps],
    description='Capture:',
    style={'description_width': 'initial'},
    layout={'width': 'max-content'}
)

output = widgets.Output()

def show_capture(change):
    output.clear_output()
    cap_data = change['new']
    img_path = Path(cap_data['path'])
    metar_path = Path(cap_data['metar']) if cap_data.get('metar') else None
    
    with output:
        if metar_path and metar_path.exists():
            print(f"METAR: {metar_path.read_text().strip()}\n")
        
        if img_path.exists():
            display(Image(filename=str(img_path), width=800))
        else:
            print(f"Image not found at {img_path}")

dropdown.observe(show_capture, names='value')
display(dropdown)
display(output)

if job_caps:
    show_capture({'new': job_caps[0]})